# Titanic Survival Predictor
1. Importing required libraries, and loading required data.
2. EDA, understand data by obersving trends, missing values and outliers.
3. Data Preprocessing, and Feature engineering -> whatever that means.
    This involves checking for missing data points, create required features from the available features.
4. Model building,
    Models to try (all in sklearn):

    Logistic Regression

    K-Nearest Neighbors (KNN)

    Decision Tree

    Random Forest

    Support Vector Machines (SVM)

    Gradient Boosting (like XGBoost)

    Evaluate using:

    Accuracy

    Precision / Recall / F1 Score

    Confusion Matrix

    Cross-validation (cross_val_score)
    
    ROC-AUC Curve


In [450]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [451]:
def load_data(file_name):
    file_path = os.path.join('..', 'data', file_name)
    if os.path.exists(file_path) != True:
        raise FileNotFoundError("Given path does not exist! = {}", format(file_path))
    else:
        df = pd.read_csv(file_path)
    return df


In [452]:
file_name = 'train.csv'
df = load_data(file_name)


In [453]:
# Pclass = socio-economic status, 1st class, 2nd class, 3rd class
# # of Siblings or spouses in ship
# # of parents or children in ship
# embarked, place of boarding, S = Southampton, Q = Queenstown, C = Cherbourg
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Data Preprocessing
1. Label Encoding
    Label Encoders are used for target variables, but not features, they can be used for ordinal features, i.e., which have an inherent order in them, but for random strings using labelling causes a bias due to numeric ordering.

2. Filling null values

In [454]:
# Hence One-Hot encoder, or pd.get_dummies can be used
# default ouput is bool for me, hence manually changing it to int
df_enc = pd.get_dummies(df, columns=['Embarked', 'Sex'], dtype=int)
df_enc.head()


,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,0,0,1,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,1,0,0,1,0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,0,0,1,1,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,0,0,1,1,0
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,NaN,0,0,1,0,1


In [455]:
# feature engineering, extracting required features.
features = df_enc.drop(columns=['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin'])
target = df_enc['Survived']
# dealing with nulls
features.isna().sum()
# returns the number of nulls per column
np.nanmean(features['Age'])


29.69911764705882

In [456]:
# we observe that 'Age' only has nulls in the feature set
# we use fillna, and replace it with a statistical term, overall not effecting the distribution
# or we can drop the row in case of scarce nulls
# for important rows we can use a model to predict the value of the null by observing patterns

# this is chained assignment and can cause errors in pandas due to features['Age'] only giving out a copy, hence not changing the actual dataframe.
features.loc[:, 'Age'] = features['Age'].fillna(np.nanmean(features['Age']))
# using loc makes sure you are modifying the original dataframe.
features.isna().sum()


Pclass        0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_C    0
Embarked_Q    0
Embarked_S    0
Sex_female    0
Sex_male      0
dtype: int64

In [461]:
# scaling to improve the accuracy of models such as SVMs which aren't scale invariant.
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# features_scaled = scaler.fit_transform(features)

# but since we generally do not have the train data, we will fit_transform train data, and use the fit model to transform the test data


In [462]:
# train test split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.3)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
# print(features.head())
# print(target.head())


In [463]:
# importing models
# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

log_reg_model = LogisticRegression()
knn_model = KNeighborsClassifier()
tree_model = DecisionTreeClassifier()
SVC_model = SVC()
NB_model = GaussianNB()
forest_model = RandomForestClassifier()


In [464]:
log_reg_model.fit(x_train, y_train)
knn_model.fit(x_train, y_train)
tree_model.fit(x_train, y_train)
SVC_model.fit(x_train, y_train)
NB_model.fit(x_train, y_train)
forest_model.fit(x_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [465]:
print(f"Accuracy logisticRegression: \n {log_reg_model.score(x_test, y_test)*100} %")
print(f"Accuracy KNN: \n {knn_model.score(x_test, y_test)*100} %")
print(f"Accuracy DecisionTree: \n {tree_model.score(x_test, y_test)*100} %")
print(f"Accuracy SVC: \n {SVC_model.score(x_test, y_test)*100} %")
print(f"Accuracy Naive Bayes: \n {NB_model.score(x_test, y_test)*100} %")
print(f"Accuracy Random Forrest Classifier: \n {forest_model.score(x_test, y_test)*100} %")


Accuracy logisticRegression: 
 78.73134328358209 %
Accuracy KNN: 
 80.97014925373134 %
Accuracy DecisionTree: 
 77.98507462686567 %
Accuracy SVC: 
 82.08955223880598 %
Accuracy Naive Bayes: 
 79.47761194029852 %
Accuracy Random Forrest Classifier: 
 81.71641791044776 %


# Peformance Metrics
TO classify the predictions, we define, TP, FP, TN, FN -> True positive, false positive, true negative, false negative.
Accuracy is the overall true predictions, 
$$\frac{TP + TN}{TP + TN + FP + FN}$$

Precision is the fraction of true positives in all positives predicted -> out of all positives, how many were true?
$$\frac{TP}{TP + FP}$$

Recall is the fraction of true positives out of all positives -> How many positives were correctly identified by the model?
$$\frac{TP}{TP + FN}$$

F1-score is the harmonic mean of both recall and precision, giving us the best of both worlds,
$$F1-score = 2 \times \frac{Recall \times Precision}{Recall + Precision}$$

In [468]:
from sklearn.metrics import classification_report
for model in [log_reg_model, knn_model, tree_model, SVC_model, NB_model, forest_model]:
    y_pred = model.predict(x_test)
    print("Accuracy ", model, '\n', classification_report(y_test, y_pred))


Accuracy  LogisticRegression() 
               precision    recall  f1-score   support

           0       0.83      0.83      0.83       168
           1       0.72      0.71      0.71       100

    accuracy                           0.79       268
   macro avg       0.77      0.77      0.77       268
weighted avg       0.79      0.79      0.79       268

Accuracy  KNeighborsClassifier() 
               precision    recall  f1-score   support

           0       0.85      0.85      0.85       168
           1       0.74      0.75      0.75       100

    accuracy                           0.81       268
   macro avg       0.80      0.80      0.80       268
weighted avg       0.81      0.81      0.81       268

Accuracy  DecisionTreeClassifier() 
               precision    recall  f1-score   support

           0       0.84      0.80      0.82       168
           1       0.69      0.75      0.72       100

    accuracy                           0.78       268
   macro avg       0.77